# ds-tutor Example: House Prices

This notebook demonstrates how to use `DsTutor` from `ds-tutor` on the House Prices Advanced Regression Techniques dataset.

In [26]:
import os

import kagglehub
import pandas as pd

In [27]:
# 1. Download the competition dataset files to your local cache
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

# 2. Load the downloaded CSVs into pandas DataFrames
df_train = pd.read_csv(os.path.join(path, 'train.csv'))
df_test = pd.read_csv(os.path.join(path, 'test.csv'))

### Inspecting the Data
Let's see what the data looks like and separate our target variable `SalePrice`.

In [28]:
X_train = df_train.drop(columns=['SalePrice', 'Id'])
y_train = df_train['SalePrice']

X_train.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal


### Evaluating Data with DsTutor

Here, we initialize `DsTutor`. `DsTutor` will inspect `X_train` and display helpful guidance directly in the notebook if it detects issues like missing data or unscaled features.

In [30]:
from ds_tutor import DsTutor

tutor = DsTutor()

# This will trigger the ds-tutor insights.
# After seeing the advice, we can construct our sklearn Pipeline accordingly!
tutor.evaluate(X_train)

### 🧑‍🏫 ds-tutor: Missing Value Check

**Missing Data Detected.**
Your dataset contains missing values (NaNs). Most scikit-learn algorithms (e.g., SVM, Random Forest, Logistic Regression) cannot handle missing values and will throw an error during training.
Dropping rows reduces your training data size, which can harm performance. Imputation (filling in the missing values) is usually preferred.


        For mean imputation, a missing value $x_{i,j}$ (where $i$ is the sample, $j$ is the feature)
        is replaced by the mean of the observed values in feature $j$:
        $$ \hat{x}_{i,j} = \mu_j = \frac{1}{N_{obs}} \sum_{k \in \text{observed}} x_{k,j} $$
        


        ```python
        from sklearn.impute import SimpleImputer
        from sklearn.pipeline import Pipeline

        # Add SimpleImputer to your pipeline
        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),  # strategies: 'mean', 'median', 'most_frequent'
            ('model', YourModelHere())
        ])
        ```
        

*Could not display visual: Mime type rendering requires nbformat>=4.2.0 but it is not installed*

...


        In Gradient Descent, the weight update rule is:
        $$w_j := w_j - \alpha \frac{\partial J}{\partial w_j}$$
        If feature $x_j$ is on a massive scale, the gradient dominates the updates, 
        causing the algorithm to oscillate and struggle to converge.
        


        ```python
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler
        from sklearn.linear_model import LogisticRegression
    
        # Added StandardScaler to your pipeline
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', LogisticRegression())
        ])```
        

*Could not display visual: Mime type rendering requires nbformat>=4.2.0 but it is not installed*

---

ValueError: Input contains NaN

### Training a simple model and diagnosing it
Let's build a very basic pipeline to handle missing values and scaling so we can train a `RandomForestRegressor`. Then we will use `DsTutor` to evaluate how well it predicts `SalePrice`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# We will use only numeric columns for simplicity in this quick test
numeric_cols = X_train.select_dtypes(include=['number']).columns
X_numeric = X_train[numeric_cols]

# Split validation data to test the model
X_tr, X_val, y_tr, y_val = train_test_split(X_numeric, y_train, test_size=0.2, random_state=42)

# Build a basic model
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=50, random_state=42))
])

pipeline.fit(X_tr, y_tr)

# Use DsTutor to diagnose the model's performance!
tutor.diagnose_regression_model(pipeline, X_val, y_val)